In [24]:
import pandas as pd
import numpy as np

In [25]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Aya Nagar, Delhi - IMD.xlsx",skiprows=16)

In [26]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [27]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (507, 10)


In [28]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene']
Dropped rows (>70% NaN): 59
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
dtype: int64


In [29]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

Dropped rows (>70% outliers): 11


In [30]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (437, 9)
   From Date    To Date   PM2.5    PM10     NO    NO2    NOx    CO  Ozone
0 2025-01-01 2025-02-01   106.4  121.09   8.53  15.35  14.87  0.54  54.45
1 2025-02-01 2025-03-01  141.02  164.48   16.2  15.87  21.18  0.85  49.22
2 2025-03-01 2025-04-01  183.31  254.44  43.77  28.65  22.99  0.98  46.26
3 2025-04-01 2025-05-01  206.62  288.44  40.45  46.56  22.99  0.98  43.95
4 2025-05-01 2025-06-01  114.26  158.47      7  19.47  15.84  0.43  35.30


In [31]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [32]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone
0,2025-01-01,2025-02-01,106.4,121.09,8.53,-1.178244,-1.225747,-1.727595,-0.007985
1,2025-02-01,2025-03-01,141.02,164.48,16.2,-1.115980,-0.191810,-0.494976,-0.324751
2,2025-03-01,2025-04-01,183.31,254.44,43.77,0.414279,0.104771,0.021928,-0.504029
3,2025-04-01,2025-05-01,206.62,288.44,40.45,2.558798,0.104771,0.021928,-0.643940
4,2025-05-01,2025-06-01,114.26,158.47,7,-0.684921,-1.066806,-2.164975,-1.167845
...,...,...,...,...,...,...,...,...,...
432,2025-07-11,2025-08-11,18.24,46.05,0,0.027523,0.104771,0.021928,0.001100
433,2025-08-11,2025-09-11,18.24,46.05,0,0.027523,0.104771,0.021928,0.001100
434,2025-09-11,2025-10-11,18.24,46.05,0,0.027523,0.104771,0.021928,0.001100
435,2025-10-11,2025-11-11,18.24,46.05,0,0.027523,0.104771,0.021928,0.001100


In [33]:
df.to_excel('Aayanagar2025.xlsx', index=False)